# Devoir1 : Pipeline appliqué à un document PDF en arabe

## 1. Importation des bibliothèques

In [1]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

## 2. Données textuelles : corpus de textes arabes

In [2]:
import pdfplumber

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            if page.extract_text():
                text += page.extract_text() + "\n"
    return text

In [3]:
# Corpus de textes arabes issus de différents domaines
# Sources : articles de presse, encyclopédies et descriptions culturelles en langue arabe

text_brut = """
تعدّ اللغة العربية من أقدم اللغات في العالم وأكثرها انتشاراً، إذ يتحدث بها أكثر من أربعمائة مليون شخص في العالم.
تنتشر اللغة العربية في منطقة الشرق الأوسط وشمال أفريقيا، وتُستخدم لغةً رسمية في اثنتين وعشرين دولة.
كما تحتل مكانة مركزية في الثقافة الإسلامية باعتبارها لغة القرآن الكريم.

تتميّز اللغة العربية بنظامها الصرفي الغني، حيث تُشتق كلمات كثيرة من جذور ثلاثية أو رباعية.
يمنح هذا النظام الاشتقاقي اللغة قدراً كبيراً من الدقة والمرونة في التعبير.
وقد أسهم العلماء العرب تاريخياً في إثراء الفلسفة والطب والرياضيات والفلك.

يشهد العالم العربي تنوعاً ثقافياً واجتماعياً بارزاً، يتجلى في العادات والتقاليد والفنون والموسيقى.
تتعدد اللهجات المحكية من المغرب إلى الخليج، غير أن الفصحى تظل الرابط المشترك بين هذه الشعوب.
وتُعدّ المدن العربية الكبرى كالقاهرة وبيروت وبغداد مراكز فكرية وثقافية عريقة.

على الصعيد الأدبي، تزخر الثقافة العربية بإرث أدبي غني يمتد من الشعر الجاهلي إلى الرواية المعاصرة.
ومن أبرز الروائيين العرب نجيب محفوظ الحائز على جائزة نوبل للآداب عام ألف وتسعمائة وثمانية وثمانين.
تتناول رواياته قضايا المجتمع المصري والهوية الإنسانية والصراع بين القيم التقليدية والحداثة.

في مجال التكنولوجيا، تشهد الدول العربية تحولاً رقمياً ملحوظاً في قطاعات التعليم والاقتصاد والإعلام.
وتسعى حكومات عديدة إلى تطوير البنية التحتية الرقمية ودعم الابتكار التكنولوجي.
كما يزداد الاهتمام بالذكاء الاصطناعي ومعالجة اللغات الطبيعية لخدمة اللغة العربية.
"""


## 3. Prétraitement du texte

Le texte extrait depuis un document peut contenir souvent :
- des espaces inutiles ;
- des chiffres ;
- de la ponctuation ;
- des mots très fréquents peu informatifs (stopwords).

L'objectif est donc de nettoyer le texte avant l'analyse.

In [5]:
# Liste simple de stopwords arabes
stopwords_ar = {
    "في", "من", "على", "إلى", "عن", "أن", "إن", "ما", "لا", "لم", "لن",
    "و", "أو", "هو", "هي", "هم", "هن", "هذا", "هذه", "ذلك", "تلك",
    "كان", "كانت", "يكون", "يمكن", "كما", "أي", "اي", "ايضا", "أيضا",
    "أكثر", "اكثر", "أقل", "اقل", "بعد", "قبل", "كل", "بعض", "قد", "لقد",
    "مع", "بين", "لدى", "هناك", "هنا", "تم", "به", "بها", "فيه", "فيها",
    "عليه", "عليها", "حتى", "اذا", "إذا", "ثم", "بل", "او", "الى"
}

def normalize_arabic(text):
    text = str(text)
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي")
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")
    text = text.replace("ة", "ه")
    text = text.replace("ـ", "")
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)  # diacritiques
    return text

def clean_arabic_text(text):
    text = normalize_arabic(text)
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)  # garder lettres arabes + espaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_text(text):
    return text.split()

def remove_stopwords(tokens):
    return [tok for tok in tokens if tok not in stopwords_ar and len(tok) > 1]

In [6]:
text_nettoye = clean_arabic_text(text_brut)
tokens = tokenize_text(text_nettoye)
tokens_clean = remove_stopwords(tokens)

print("=== Aperçu du texte nettoyé ===")
print(text_nettoye[:300])
print(f"\nNombre de tokens : {len(tokens)}")
print(f"Tokens sans stopwords : {len(tokens_clean)}")

=== Aperçu du texte nettoyé ===
تعد اللغه العربيه من اقدم اللغات في العالم واكثرها انتشارا، اذ يتحدث بها اكثر من اربعمايه مليون شخص في العالم تنتشر اللغه العربيه في منطقه الشرق الاوسط وشمال افريقيا، وتستخدم لغه رسميه في اثنتين وعشرين دوله كما تحتل مكانه مركزيه في الثقافه الاسلاميه باعتبارها لغه القران الكريم تتميز اللغه العربيه بن

Nombre de tokens : 205
Tokens sans stopwords : 180


## 4. Réduction linguistique : stemming léger

Pour l'arabe, on applique ici un **stemming léger simplifié**, qui retire certains préfixes et suffixes fréquents.  
Ce n'est pas une lemmatisation complète, mais cela permet déjà de réduire certaines variantes lexicales.

In [7]:
def light_stem(word):
    prefixes = ["وال", "بال", "كال", "فال", "لل", "ال"]
    suffixes = ["يات", "ات", "ون", "ين", "ان", "ها", "هم", "هن", "كما", "كم", "نا"]

    original = word

    for p in prefixes:
        if word.startswith(p) and len(word) > len(p) + 2:
            word = word[len(p):]
            break

    for s in suffixes:
        if word.endswith(s) and len(word) > len(s) + 2:
            word = word[:-len(s)]
            break

    if len(word) < 3:
        return original

    return word

In [8]:
stems = [light_stem(tok) for tok in tokens_clean]

print("=== 30 premiers mots après stemming léger ===")
print(stems[:30])

=== 30 premiers mots après stemming léger ===
['تعد', 'لغه', 'عربيه', 'اقدم', 'لغات', 'عالم', 'واكثر', 'انتشارا،', 'اذ', 'يتحدث', 'اربعمايه', 'ملي', 'شخص', 'عالم', 'تنتشر', 'لغه', 'عربيه', 'منطقه', 'شرق', 'اوسط', 'وشمال', 'افريقيا،', 'وتستخدم', 'لغه', 'رسميه', 'اثنت', 'وعشر', 'دوله', 'تحتل', 'مكانه']


## 5. Représentation vectorielle avec TF-IDF

In [9]:
bad_stems = {
    "ان", "علي", "الى", "الي", "هذا", "هذه", "ذلك", "تلك",
    "هناك", "هنا", "لكن", "اكثر", "اقل", "احد", "اخر", "اخره",
    "لان", "لقد", "قد", "ثم", "مع", "من", "في", "عن", "كل", "بين"
}

stems_final = [s for s in stems if len(s) > 2 and s not in bad_stems]

document_prepare = " ".join(stems_final)

# Corpus de référence pour le calcul TF-IDF
corpus = [
    document_prepare,
    "اللغه العربيه لغه قديمه تنتشر في الشرق الاوسط وشمال افريقيا",
    "الادب العربي يشمل الشعر والرواية والقصة القصيرة",
    "التكنولوجيا الرقميه والذكاء الاصطناعي يخدمان اللغه العربيه",
    "الثقافه العربيه تتنوع بين المدن من المغرب الى الخليج"
]

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.80
)

X_tfidf = vectorizer.fit_transform(corpus)

df_tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)

top_terms = df_tfidf.iloc[0].sort_values(ascending=False).head(20)

top_terms_df = top_terms.reset_index()
top_terms_df.columns = ["Mot", "Score"]

top_terms_df

,Mot,Score
0,عربيه,0.334283
1,لغه,0.269697
2,لغه عربيه,0.191019
3,عالم,0.143264
4,ثقافه,0.095509
5,عرب,0.095509
6,غني,0.095509
7,لغات,0.095509
8,ادبي,0.095509
9,وعشر دوله,0.047755


In [10]:
top_terms = df_tfidf.iloc[0].sort_values(ascending=False).head(20)

print("=== Top 20 des mots les plus importants selon TF-IDF ===")
print(top_terms)

=== Top 20 des mots les plus importants selon TF-IDF ===
عربيه            0.334283
لغه              0.269697
لغه عربيه        0.191019
عالم             0.143264
ثقافه            0.095509
عرب              0.095509
غني              0.095509
لغات             0.095509
ادبي             0.095509
وعشر دوله        0.047755
مدن عربيه        0.047755
وثقافيه عريقه    0.047755
اعلام            0.047755
يتجلي            0.047755
وعشر             0.047755
اداب             0.047755
ابرز روايي       0.047755
الف              0.047755
الف وتسعمايه     0.047755
اعلام وتسعي      0.047755
Name: 0, dtype: float64


## 6. Lecture synthétique des résultats

À ce stade, on peut observer que :
- le texte a bien été nettoyé et normalisé ;
- les stopwords ont été supprimés ;
- le stemming léger a réduit certaines variantes morphologiques ;
- TF-IDF permet d'identifier les mots les plus représentatifs du corpus.

Dans ce corpus, on remarque la présence de vocabulaire lié :
- à la **langue arabe** et à son histoire ;
- à la **littérature et culture** arabes ;
- à la **technologie** et au numérique.

In [11]:
resume = pd.DataFrame({
    "Etape": [
        "Texte brut",
        "Texte nettoye",
        "Nombre total de tokens",
        "Nombre de tokens sans stopwords",
        "Nombre de stems",
        "Nombre de termes TF-IDF"
    ],
    "Valeur": [
        text_brut[:120] + "...",
        text_nettoye[:120] + "...",
        len(tokens),
        len(tokens_clean),
        len(stems),
        len(vectorizer.get_feature_names_out())
    ]
})

resume

,Etape,Valeur
0,Texte brut,\nتعدّ اللغة العربية من أقدم اللغات في العالم ...
1,Texte nettoye,تعد اللغه العربيه من اقدم اللغات في العالم واك...
2,Nombre total de tokens,205
3,Nombre de tokens sans stopwords,180
4,Nombre de stems,180
5,Nombre de termes TF-IDF,376
